##### Ideal QSVM Optimization - Lung Cancer (Full Dataset)

In [33]:
# ===================================================================
# QSVM HYPERPARAMETER OPTIMIZATION - LUNG CANCER (FULL DATASET)
# ===================================================================

"""
This notebook systematically explores QSVM hyperparameters to identify
optimal configurations for the Lung Cancer dataset.

Experiments:
- Feature counts: [4, 6, 8, 10] → Qubit count
- Reps: [1, 2, 3] → Circuit depth
- Entanglement: ['linear', 'full'] → Feature correlations
- Shots: [256, 512, 1024] → Sampling precision

Goal: Find configuration that maximizes test accuracy on full dataset (32 samples).
"""

"\nThis notebook systematically explores QSVM hyperparameters to identify\noptimal configurations for the Lung Cancer dataset.\n\nExperiments:\n- Feature counts: [4, 6, 8, 10] → Qubit count\n- Reps: [1, 2, 3] → Circuit depth\n- Entanglement: ['linear', 'full'] → Feature correlations\n- Shots: [256, 512, 1024] → Sampling precision\n\nGoal: Find configuration that maximizes test accuracy on full dataset (32 samples).\n"

In [34]:
!pip install qiskit qiskit-machine-learning qiskit-aer

In [ ]:
# Imports
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, classification_report, balanced_accuracy_score
from scipy.stats import chi2_contingency

In [36]:
# Qiskit Imports
from qiskit.circuit.library import ZZFeatureMap
from qiskit.primitives import StatevectorSampler as Sampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.utils import algorithm_globals

In [37]:
# Set seed for reproducibility
algorithm_globals.random_seed = 12345
np.random.seed(12345)

##### Data Loading and Preprocessing

In [ ]:
# Load Lung Cancer Dataset
lung_cancer_column_names = ['label'] + [f'attr_{i}' for i in range(1, 57)]
file_path_lung = r'C:\Users\User\Documents\MyProjects\AI_Projects\quantum-svm-generalization-study\data\lung+cancer\lung-cancer.data'
# file_path_lung = '/content/sample_data/lungcancer/lung-cancer.data'
# file_path_lung = '/content/lung-cancer.data' # Adjusted path for Colab upload

# Read data, treating "?" as missing values
df_lung = pd.read_csv(file_path_lung, header=None, names=lung_cancer_column_names, na_values=['?'])

print(f"Original shape of Lung Cancer data: {df_lung.shape}")
print(f"Missing values: {df_lung.isnull().sum().sum()}")

In [ ]:
# Mode imputation for missing values
modes = df_lung.mode().iloc[0]
df_lung.fillna(modes, inplace=True)

# Then check if all Nan are gon
print(f"Total missing values after imputation: {df_lung.isnull().sum().sum()}\n")

In [ ]:
# Target Binarization: Class 1 vs Others (2,3)
df_lung['label_binary'] = df_lung['label'].apply(lambda x: 0 if x == 1 else 1)

# print("Class distribution:")
# print(df_lung['label_binary'].value_counts())
# print()

In [ ]:
# Separate Features & Target
X_lung = df_lung.drop(['label', 'label_binary'], axis=1)
y_lung_binary = df_lung['label_binary']

In [ ]:
X_train_lc, X_test_lc, y_train_lc, y_test_lc = train_test_split(
    X_lung, y_lung_binary, test_size=0.3, random_state=42, stratify=y_lung_binary
)

print(f"Training samples: {X_train_lc.shape[0]}")
print(f"Test samples: {X_test_lc.shape[0]}\n")

In [ ]:
# One-Hot Encoding
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_lc_encoded = pd.DataFrame(
    encoder.fit_transform(X_train_lc),
    columns=encoder.get_feature_names_out()
)
X_test_lc_encoded = pd.DataFrame(
    encoder.transform(X_test_lc),
    columns=encoder.get_feature_names_out()
)

print(f"Encoded features: {X_train_lc_encoded.shape[1]}")

In [ ]:
# Feature Selection - Cramer's V
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    if min((kcorr-1), (rcorr-1)) == 0: return 0
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

cramers_scores = {col: cramers_v(X_train_lc_encoded[col], y_train_lc) for col in X_train_lc_encoded.columns}
cramers_series = pd.Series(cramers_scores).sort_values(ascending=False)

N_FEATURES_TO_SELECT = 10 
top_features = cramers_series.head(N_FEATURES_TO_SELECT).index.tolist()

X_train_lc_final = X_train_lc_encoded[top_features]
X_test_lc_final = X_test_lc_encoded[top_features]

print("--- Data Preprocessing Complete ---")
print(f"Final training data shape: {X_train_lc_final.shape}")
print(f"Final testing data shape: {X_test_lc_final.shape}\n")


##### Optimization Setup

In [ ]:
# ===================================================================
# OPTIMIZATION EXPERIMENT CONFIGURATION
# ===================================================================
print("="*80)
print("HYPERPARAMETER OPTIMIZATION GRID")
print("="*80)

# Define parameter grid
FEATURE_COUNTS = [4, 6, 8, 10]  # Number of features (qubits)
REPS = [1, 2, 3]  # Circuit depths
ENTANGLEMENTS = ['linear', 'full']  # Entanglement topologies
SHOTS = [256, 512, 1024]  # Sampling precision

# Calculate total experiments
total_experiments = len(FEATURE_COUNTS) * len(REPS) * len(ENTANGLEMENTS) * len(SHOTS)
print(f"Total configurations to test: {total_experiments}\n")

# Results storage
results = []

##### Optimization Loop

In [ ]:
# ===================================================================
# OPTIMIZATION LOOP (WITH HYPERPARAMETER TUNING)
# ===================================================================

experiment_num = 0

for n_features, reps, entanglement, shots in product(FEATURE_COUNTS, REPS, ENTANGLEMENTS, SHOTS):
    experiment_num += 1
    
    print("="*80)
    print(f"EXPERIMENT {experiment_num}/{total_experiments}")
    print(f"  Features (Qubits): {n_features}")
    print(f"  Reps: {reps}")
    print(f"  Entanglement: {entanglement}")
    print(f"  Shots: {shots}")
    print("="*80)
    
    try:
        # --- Select Top N Features ---
        top_features = cramers_series.head(n_features).index.tolist()
        X_train_selected = X_train_lc_encoded[top_features]
        X_test_selected = X_test_lc_encoded[top_features]
        
        print(f"Using top {n_features} features")
        
        # --- Create Quantum Kernel ---
        fm = ZZFeatureMap(
            feature_dimension=n_features,
            reps=reps,
            entanglement=entanglement
        )
        
        sampler = Sampler(default_shots=shots)
        fidelity = ComputeUncompute(sampler=sampler)
        qkernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=fm)
        
        # --- Compute Kernel Matrices ---
        print("Computing kernel matrices...")
        start_kernel = time.time()
        
        K_train = qkernel.evaluate(x_vec=X_train_selected.to_numpy())
        K_test = qkernel.evaluate(x_vec=X_test_selected.to_numpy(), y_vec=X_train_selected.to_numpy())
        
        kernel_time = time.time() - start_kernel
        print(f"Kernel computation: {kernel_time:.2f}s")
        
        # --- GRID SEARCH FOR OPTIMAL C ---
        print("Grid searching for optimal C...")
        start_train = time.time()
        
        param_grid = {
            'C': [0.001, 0.01, 0.1, 1, 10, 100]
        }
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        
        grid_search = GridSearchCV(
            SVC(kernel='precomputed', class_weight='balanced'),
            param_grid,
            cv=cv,
            scoring='accuracy',
            n_jobs=-1
        )
        
        grid_search.fit(K_train, y_train_lc)
        best_svm = grid_search.best_estimator_
        best_C = grid_search.best_params_['C']
        cv_score = grid_search.best_score_
        
        train_time = time.time() - start_train
        
        print(f"  → Best C: {best_C}")
        print(f"  → CV Score: {cv_score:.4f}")
        print(f"  → Training time: {train_time:.2f}s")
        
        # --- Evaluate ---
        y_train_pred = best_svm.predict(K_train)
        y_test_pred = best_svm.predict(K_test)
        
        train_acc = accuracy_score(y_train_lc, y_train_pred)
        test_acc = accuracy_score(y_test_lc, y_test_pred)
        test_bal_acc = balanced_accuracy_score(y_test_lc, y_test_pred)
        class1_recall = recall_score(y_test_lc, y_test_pred, pos_label=1)
        gen_gap = abs(train_acc - test_acc)
        
        print(f"  → Train Accuracy: {train_acc:.4f}")
        print(f"  → Test Accuracy: {test_acc:.4f}")
        print(f"  → Test Balanced Accuracy: {test_bal_acc:.4f}")
        print(f"  → Generalization Gap: {gen_gap:.4f}")
        print(f"  → Class 1 Recall: {class1_recall:.4f}\n")
        
        # --- Store Results ---
        results.append({
            'N_Features': n_features,
            'Reps': reps,
            'Entanglement': entanglement,
            'Shots': shots,
            'Best_C': best_C,
            'CV_Score': cv_score,
            'Train_Acc': train_acc,
            'Test_Acc': test_acc,
            'Test_Balanced_Acc': test_bal_acc,
            'Class1_Recall': class1_recall,
            'Generalization_Gap': gen_gap,
            'Kernel_Time': kernel_time,
            'Train_Time': train_time,
            'Total_Time': kernel_time + train_time,
            'Status': 'Success'
        })
        
    except Exception as e:
        print(f"ERROR: {str(e)}\n")
        results.append({
            'N_Features': n_features,
            'Reps': reps,
            'Entanglement': entanglement,
            'Shots': shots,
            'Best_C': np.nan,
            'CV_Score': np.nan,
            'Train_Acc': np.nan,
            'Test_Acc': np.nan,
            'Test_Balanced_Acc': np.nan,
            'Class1_Recall': np.nan,
            'Generalization_Gap': np.nan,
            'Kernel_Time': np.nan,
            'Train_Time': np.nan,
            'Total_Time': np.nan,
            'Status': f'Failed: {str(e)}'
        })

##### Results Analysis

In [ ]:
# ===================================================================
# RESULTS ANALYSIS
# ===================================================================

print("\n" + "="*80)
print("OPTIMIZATION RESULTS SUMMARY")
print("="*80)

df_results = pd.DataFrame(results)

# Filter successful runs
df_success = df_results[df_results['Status'] == 'Success']

# Display all results sorted by test accuracy
df_sorted = df_success.sort_values('Test_Acc', ascending=False)
print("\nTop 10 Configurations (sorted by Test Accuracy):")
print(df_sorted[['N_Features', 'Reps', 'Entanglement', 'Shots', 'Test_Acc', 'Test_Balanced_Acc', 'Generalization_Gap']].head(10).to_string(index=False))

# Find best configuration
best_idx = df_success['Test_Acc'].idxmax()
best_config = df_success.loc[best_idx]

print("\n" + "="*80)
print("BEST CONFIGURATION")
print("="*80)
print(f"Features (Qubits): {best_config['N_Features']}")
print(f"Reps: {best_config['Reps']}")
print(f"Entanglement: {best_config['Entanglement']}")
print(f"Shots: {best_config['Shots']}")
print(f"Best C: {best_config['Best_C']}")
print(f"---")
print(f"Test Accuracy: {best_config['Test_Acc']:.4f}")
print(f"Test Balanced Accuracy: {best_config['Test_Balanced_Acc']:.4f}")
print(f"Train Accuracy: {best_config['Train_Acc']:.4f}")
print(f"Generalization Gap: {best_config['Generalization_Gap']:.4f}")
print(f"Class 1 Recall: {best_config['Class1_Recall']:.4f}")
print(f"Total Time: {best_config['Total_Time']:.1f}s")

# Save results
df_results.to_csv('qsvm_optimization_results_lungcancer.csv', index=False)
print("\n✓ Results saved to 'qsvm_optimization_results_lungcancer.csv'")

##### Best Configuration - Detailed Evaluation

In [ ]:
# ===================================================================
# BEST CONFIGURATION - DETAILED EVALUATION
# ===================================================================

print("\n" + "="*80)
print("BEST CONFIGURATION - DETAILED EVALUATION")
print("="*80)

best_idx = df_success['Test_Balanced_Acc'].idxmax()
best_row = df_success.loc[best_idx]

# Re-run best config to get classification report
n_features_best = int(best_row['N_Features'])
top_features_best = cramers_series.head(n_features_best).index.tolist()
X_train_best = X_train_lc_encoded[top_features_best]
X_test_best = X_test_lc_encoded[top_features_best]

fm_best = ZZFeatureMap(
    feature_dimension=n_features_best,
    reps=int(best_row['Reps']),
    entanglement=best_row['Entanglement']
)
sampler_best = Sampler(default_shots=int(best_row['Shots']))
fidelity_best = ComputeUncompute(sampler=sampler_best)
qkernel_best = FidelityQuantumKernel(fidelity=fidelity_best, feature_map=fm_best)

K_train_best = qkernel_best.evaluate(x_vec=X_train_best.to_numpy())
K_test_best = qkernel_best.evaluate(x_vec=X_test_best.to_numpy(), y_vec=X_train_best.to_numpy())

svm_best = SVC(kernel='precomputed', class_weight='balanced', C=best_row['Best_C'])
svm_best.fit(K_train_best, y_train_lc)

y_test_pred_best = svm_best.predict(K_test_best)

print(f"\nConfiguration:")
print(f"  Features: {n_features_best}")
print(f"  Reps: {int(best_row['Reps'])}")
print(f"  Entanglement: {best_row['Entanglement']}")
print(f"  Shots: {int(best_row['Shots'])}")
print(f"  Best C: {best_row['Best_C']}")
print(f"\nPerformance:")
print(f"  Test Accuracy: {best_row['Test_Acc']:.4f}")
print(f"  Test Balanced Accuracy: {best_row['Test_Balanced_Acc']:.4f}")
print(f"  Class 1 Recall: {best_row['Class1_Recall']:.4f}")
print(f"  Generalization Gap: {best_row['Generalization_Gap']:.4f}")
print(f"\nClassification Report (Test Set):")
print(classification_report(y_test_lc, y_test_pred_best, zero_division=0))

##### Visualizations

In [ ]:
# ===================================================================
# GENERATING VISUALIZATIONS
# ===================================================================

print("\n" + "="*80)
print("GENERATING VISUALIZATIONS")
print("="*80)

# 1. Heatmap: Test Accuracy by Features and Reps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, ent in enumerate(['linear', 'full']):
    df_ent = df_success[df_success['Entanglement'] == ent]
    
    pivot = df_ent.pivot_table(
        values='Test_Acc',
        index='Reps',
        columns='N_Features',
        aggfunc='mean'
    )
    
    sns.heatmap(
        pivot,
        annot=True,
        fmt='.3f',
        cmap='YlGnBu',
        cbar_kws={'label': 'Test Accuracy'},
        ax=axes[idx],
        vmin=0.3,
        vmax=0.8
    )
    
    axes[idx].set_title(f'Entanglement: {ent}')
    axes[idx].set_xlabel('Number of Features (Qubits)')
    axes[idx].set_ylabel('Circuit Depth (Reps)')

plt.tight_layout()
plt.savefig('heatmap_test_accuracy_lungcancer.png', dpi=300, bbox_inches='tight')
print("✓ Saved: heatmap_test_accuracy_lungcancer.png")
plt.show()

# 2. Bar Plot: Test Accuracy by Configuration
fig, ax = plt.subplots(figsize=(14, 6))

df_plot = df_success.copy()
df_plot['Config'] = df_plot.apply(
    lambda row: f"F{int(row['N_Features'])}_R{int(row['Reps'])}_{row['Entanglement'][:3]}",
    axis=1
)

df_plot_sorted = df_plot.sort_values('Test_Acc', ascending=False).head(20)

ax.bar(range(len(df_plot_sorted)), df_plot_sorted['Test_Acc'], color='steelblue')
ax.axhline(y=0.5, color='red', linestyle='--', label='Baseline (50%)')

ax.set_xticks(range(len(df_plot_sorted)))
ax.set_xticklabels(df_plot_sorted['Config'], rotation=45, ha='right')
ax.set_ylabel('Test Accuracy')
ax.set_xlabel('Configuration')
ax.set_title('QSVM Test Accuracy - Top 20 Configurations (Lung Cancer)')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('bar_test_accuracy_lungcancer.png', dpi=300, bbox_inches='tight')
print("✓ Saved: bar_test_accuracy_lungcancer.png")
plt.show()

# 3. Scatter: Accuracy vs Computation Time
fig, ax = plt.subplots(figsize=(10, 6))

colors = {'linear': 'blue', 'full': 'orange'}
markers = {1: 'o', 2: 's', 3: '^'}

for ent in ['linear', 'full']:
    for rep in [1, 2, 3]:
        df_subset = df_success[
            (df_success['Entanglement'] == ent) & 
            (df_success['Reps'] == rep)
        ]
        
        ax.scatter(
            df_subset['Total_Time'],
            df_subset['Test_Acc'],
            c=colors[ent],
            marker=markers[rep],
            s=df_subset['N_Features'] * 30,
            alpha=0.7,
            label=f'{ent}, reps={rep}'
        )

ax.set_xlabel('Total Computation Time (seconds)')
ax.set_ylabel('Test Accuracy')
ax.set_title('Accuracy vs Computational Cost (Lung Cancer)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('scatter_accuracy_vs_time_lungcancer.png', dpi=300, bbox_inches='tight')
print("✓ Saved: scatter_accuracy_vs_time_lungcancer.png")
plt.show()

# 4. Line Plot: Effect of Feature Count
fig, ax = plt.subplots(figsize=(10, 6))

for ent in ['linear', 'full']:
    for rep in [1, 2, 3]:
        df_subset = df_success[
            (df_success['Entanglement'] == ent) & 
            (df_success['Reps'] == rep)
        ].sort_values('N_Features')
        
        if len(df_subset) > 0:
            ax.plot(
                df_subset['N_Features'],
                df_subset['Test_Acc'],
                marker='o',
                label=f'{ent}, reps={rep}'
            )

ax.set_xlabel('Number of Features (Qubits)')
ax.set_ylabel('Test Accuracy')
ax.set_title('Impact of Feature Count on QSVM Performance (Lung Cancer)')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xticks([4, 6, 8, 10])

plt.tight_layout()
plt.savefig('line_feature_effect_lungcancer.png', dpi=300, bbox_inches='tight')
print("✓ Saved: line_feature_effect_lungcancer.png")
plt.show()

print("\n" + "="*80)
print("OPTIMIZATION COMPLETE!")
print("="*80)
print(f"Best Test Accuracy: {best_config['Test_Acc']:.4f}")
print(f"Best Balanced Accuracy: {best_config['Test_Balanced_Acc']:.4f}")
print(f"Baseline Accuracy (from basic run): 0.5000")
print(f"Improvement: {(best_config['Test_Acc'] - 0.5000):.4f}")